# Getting docs

In [2]:
%ls

data_processing.ipynb  raw/


In [3]:
# Numpy 2.5 docs

#https://stackoverflow.com/questions/69782728/urllib-error-httperror-http-error-403-forbidden-with-urllib-requests
import urllib.request
import zipfile

path = "raw/numpy-html"
zippath = "raw/numpy-html" + ".zip"

opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'MyApp/1.0')]
urllib.request.install_opener(opener)
urllib.request.urlretrieve("https://numpy.org/doc/2.5/numpy-html.zip", zippath)

#https://stackoverflow.com/questions/3451111/unzipping-files-in-python

with zipfile.ZipFile(zippath, 'r') as zip_ref:
    zip_ref.extractall(path)

# Parsing

In [4]:
from pathlib import Path
from bs4 import BeautifulSoup
from hashlib import sha256

path = Path("raw/numpy-html")

ids = []
documents = []
metadatas = []

for html_file in path.rglob("*.html"):
    
    with html_file.open("r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else ""

    #print(html_file)

    #print(soup)

    try:
        text = soup.find('article').text # Get article
    
        text = list(filter(None, text.split("\n"))) # Remove \n
    
        if (text[-1] == "Go BackOpen In Tab"): # Sometimes last line is this. Remove it
            text = text[:-1]
    
        text = "\n".join(text) # Add linebreks back in
    
        ids.append(sha256(text.encode('utf-8')).hexdigest())
        documents.append(text)
        metadatas.append({"file":str(html_file)})
    except:
        print("Error on file:", str(html_file)) 


Error on file: raw/numpy-html/search.html
Error on file: raw/numpy-html/lite/index.html
Error on file: raw/numpy-html/_static/webpack-macros.html
Error on file: raw/numpy-html/lite/tree/index.html
Error on file: raw/numpy-html/lite/lab/index.html
Error on file: raw/numpy-html/lite/repl/index.html
Error on file: raw/numpy-html/lite/edit/index.html
Error on file: raw/numpy-html/lite/consoles/index.html
Error on file: raw/numpy-html/lite/notebooks/index.html
Error on file: raw/numpy-html/lite/doc/tree/index.html
Error on file: raw/numpy-html/lite/doc/workspaces/index.html
Error on file: raw/numpy-html/lite/lab/tree/index.html
Error on file: raw/numpy-html/lite/lab/workspaces/index.html


In [5]:
len(ids)

2669

# Building chromadb

In [6]:
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [7]:
import chromadb

chroma_client = chromadb.PersistentClient("../backend/chroma.db")

In [8]:
collection = chroma_client.get_or_create_collection(name="numpy_docs")
collection.add(ids=ids, documents=documents, metadatas=metadatas)

In [9]:
results = collection.query(query_texts=["How to reshape an np.array?"])

In [10]:
results

{'ids': [['6c376a66da51b6ae8281b2846270c331ca2da933855ec633734b10608186f86d',
   '09ef3d953f9bfb2e6e6efbc4bf6400ecf8d663ebafd411de70c70f8908b91c6d',
   '57f470832829e505f78864ba9d5e880ef0130d35a0de1a761f6f13d3fa868725',
   '22605ee10f3d6c74c48e5566d402c4b15a4023644a308997d361a7dc8ce9260d',
   '6d971107c63f4606c435032c2a085cc9dab640289bb1d3c06854321c1d30807d',
   '755c1f4dff478e4af9857d0b447f1ce0a149bd3e65f880985a282d7af4a8f1c0',
   'b54eaedd6073f39a73cc1fa8ea94dfec1b5714f1e849e86dcae173c02ef08ce2',
   '87c04e59a1f484eb75f7474bc9f774cd5b45ca62d9e9f2f9a87b03bce1eafd1d',
   '8c0c6e292c609c24f0ae8368fae87c0dda8a66419a0feefc85cf8abbded06906',
   'ab2802f336fe8d2ae7198c1538ccd7527716e2eb8e0a4558624ca09e6bcb1ea8']],
 'embeddings': None,
 'documents': [['numpy.ndarray.shape#\nattribute\nndarray.shape#\nTuple of array dimensions.\nThe shape property is usually used to get the current shape of an array,\nbut may also be used to reshape the array in-place by assigning a tuple of\narray dimensions